In [1]:
import pandas as pd
import sys, os
from tqdm import tqdm
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.colors as mcolors

sys.path.append(os.path.abspath("../src"))

from utils.wiki import (
    get_template_links,
    get_revision_diff,
    get_revisions,
    format_revisions,
    select_interesting_revisions,
    get_revisions_cached,

)
from utils.poly import get_event_slugs_paginated, get_price_series_from_slug
from utils.data_cleaning import validate_and_resample_panel
from utils.text import WIKI_STOP_WORDS
from utils.text.cleaning import (
    extract_added_text,
    remove_wiki_usernames,
    looks_like_username,
    contains_keywords,
)
from utils.text.tokenizer import spacy_tokenizer
from utils.text.nmf import build_text_nmf_features_safe

from targets import add_directional_targets

In [ ]:
''' commented out until ready to retrieve new wiki data'''
titles = get_template_links("Template:Campaignbox 2026 Iran war")
from tqdm import tqdm
all_revisions = []
features = pd.DataFrame()
for i in tqdm(titles):
    rev = get_revisions_cached(i, talk=True, start="2026-02-27", end="2026-06-01",)
    rev_diff = select_interesting_revisions(rev)

    if not rev.empty:
        all_revisions.append(rev)
    if not rev_diff.empty:
        all_revisions.append(rev_diff)

100%|██████████| 116/116 [01:32<00:00,  1.26it/s]


In [4]:
from pathlib import Path
import pandas as pd

cache_dir = Path("data/wiki_cache/revisions")

dfs = [
    pd.read_parquet(path)
    for path in cache_dir.glob("*.parquet")
]

all_revisions = pd.concat(dfs, ignore_index=True)
print(all_revisions.shape)
all_revisions.head()

(9977, 14)


,revid,parentid,user,timestamp,size,comment,temp,page_title,comment_len,has_reply,has_revert,size_change,userhidden,commenthidden
0,1340908147,1340418281,Vice regent,2026-02-28 11:23:14+00:00,46800,/* Iranian govt is an unreliable source for th...,None,Talk:2026 Iran massacres,68,1,0,NaN,NaN,NaN
1,1340908641,1340908147,Vice regent,2026-02-28 11:28:01+00:00,47046,/* Image for casualties section */ Reply,None,Talk:2026 Iran massacres,40,1,0,246.0,NaN,NaN
2,1340908661,1340908641,Vice regent,2026-02-28 11:28:15+00:00,47047,/* Image for casualties section */,None,Talk:2026 Iran massacres,34,0,0,1.0,NaN,NaN
3,1340936893,1340908661,Boud,2026-02-28 16:08:19+00:00,47050,/* Table */ replace deprecated archive,None,Talk:2026 Iran massacres,38,0,0,3.0,NaN,NaN
4,1340937533,1340936893,Boud,2026-02-28 16:13:37+00:00,47165,/* Table */ archive for https://iranvictims.co...,None,Talk:2026 Iran massacres,80,0,0,115.0,NaN,NaN


In [5]:


escalation_words = [
    "attack", "strike", "missile", "nuclear", "war", "bomb", "military"
]

uncertainty_words = [
    "report", "unconfirmed", "alleged", "rumor", "claim", "possible"
]

all_revisions["comment"] = all_revisions["comment"].fillna("")

all_revisions["has_escalation"] = all_revisions["comment"].apply(
    lambda x: contains_keywords(x, escalation_words)
)

all_revisions["has_uncertainty"] = all_revisions["comment"].apply(
    lambda x: contains_keywords(x, uncertainty_words)
)

all_revisions["seen_before_global"] = all_revisions["user"].duplicated()
all_revisions["new_editor_global"] = (~all_revisions["seen_before_global"]).astype(int)

all_revisions["comment_len"] = all_revisions["comment"].str.len()
all_revisions["has_reply"] = all_revisions["comment"].str.contains("Reply", case=False).astype(int)
all_revisions["has_revert"] = all_revisions["comment"].str.contains("revert", case=False, regex=True).astype(int)

all_revisions["size_change"] = all_revisions["size"].diff().abs()

all_revisions["timestamp"] = pd.to_datetime(all_revisions["timestamp"], utc=True)
all_revisions = all_revisions.sort_values("timestamp")
all_revisions["timestamp"] = all_revisions["timestamp"].dt.floor("h")

features = all_revisions.groupby("timestamp").agg(
    edits=("user", "count"),
    unique_editors=("user", "nunique"),
    new_editors=("new_editor_global", "sum"),
    num_seen_before_global=("seen_before_global", "sum"),
    total_comment_len=("comment_len", "sum"),
    avg_comment_len=("comment_len", "mean"),
    num_replies=("has_reply", "sum"),
    num_reverts=("has_revert", "sum"),
    num_escalation_comments=("has_escalation", "sum"),
    num_uncertainty_comments=("has_uncertainty", "sum"),
    total_size_change=("size_change", "sum"),
    max_size_change=("size_change", "max"),
).reset_index().sort_values("timestamp")
print(features.shape)
features.head()


(1623, 13)


,timestamp,edits,unique_editors,new_editors,num_seen_before_global,total_comment_len,avg_comment_len,num_replies,num_reverts,num_escalation_comments,num_uncertainty_comments,total_size_change,max_size_change
0,2026-02-27 02:00:00+00:00,2,2,1,1,55,27.500000,0,0,0,0,4058.0,3875.0
1,2026-02-27 04:00:00+00:00,6,3,1,5,244,40.666667,2,0,0,0,47824.0,47101.0
2,2026-02-27 08:00:00+00:00,2,1,1,1,176,88.000000,0,0,0,0,21755.0,20733.0
3,2026-02-27 13:00:00+00:00,1,1,1,0,66,66.000000,1,0,0,0,4419.0,4419.0
4,2026-02-27 14:00:00+00:00,1,1,1,0,43,43.000000,1,0,0,0,278.0,278.0


In [6]:
market_dfs = []
slugs = get_event_slugs_paginated("Iran", pages=10, limit=100)

for slug in tqdm(slugs):
    slug = slug["slug"]
    prices, meta = get_price_series_from_slug(
        slug=slug,
        days=60,
        chunk_days=7,
        interval="1h",
        fidelity=60,
    )
    market_df = prices.copy()
    market_df["market_slug"] = slug
    market_df["market_title"] = meta["market_question"]

    # compute target within this market
    market_df = market_df.sort_values("timestamp")
    # market_df["volatility_6h"] = market_df["price"].diff().rolling(6).std()
    # market_df["target"] = market_df["volatility_6h"].shift(-1)

    market_dfs.append(market_df)

100%|██████████| 13/13 [00:25<00:00,  1.93s/it]


In [7]:
panel_df = pd.concat(market_dfs, ignore_index=True)
print(panel_df.shape)
panel_df.head()

(15047, 4)


/var/folders/1r/k6vlr74j5_j3ftfp83kg694m0000gn/T/ipykernel_88702/2191102582.py:1: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  panel_df = pd.concat(market_dfs, ignore_index=True)


,timestamp,price,market_slug,market_title
0,2026-04-07 02:00:58+00:00,0.275,will-the-iranian-regime-fall-by-the-end-of-2026,Will the Iranian regime fall before 2027?
1,2026-04-07 04:00:53+00:00,0.265,will-the-iranian-regime-fall-by-the-end-of-2026,Will the Iranian regime fall before 2027?
2,2026-04-07 14:00:40+00:00,0.245,will-the-iranian-regime-fall-by-the-end-of-2026,Will the Iranian regime fall before 2027?
3,2026-04-07 17:00:28+00:00,0.265,will-the-iranian-regime-fall-by-the-end-of-2026,Will the Iranian regime fall before 2027?
4,2026-04-07 18:00:42+00:00,0.265,will-the-iranian-regime-fall-by-the-end-of-2026,Will the Iranian regime fall before 2027?


In [8]:
panel_clean = validate_and_resample_panel(panel_df, freq="1h")
panel_clean = panel_clean.sort_values(["market_slug", "timestamp"])

In [ ]:
# -----------------------------
# 1. Merge prices + features
# -----------------------------
# Assumes:
# panel_clean has: timestamp, market_slug, price
# features has: timestamp, edits, unique_editors, new_editors, etc.
# If features are global to all markets, merge only on timestamp.
# If features are market-specific, include market_slug and merge by market_slug too.

prices_hourly = panel_clean.copy()
prices_hourly["timestamp"] = pd.to_datetime(prices_hourly["timestamp"], utc=True)

features_hourly = (
    features.copy()
    .assign(timestamp=lambda x: pd.to_datetime(x["timestamp"], utc=True))
    .set_index("timestamp")
    .resample("1h")
    .sum()
    .fillna(0)
    .reset_index()
)

df = pd.merge_asof(
    prices_hourly.sort_values("timestamp"),
    features_hourly.sort_values("timestamp"),
    on="timestamp",
    direction="backward"
)

df = df.sort_values(["market_slug", "timestamp"]).copy()
df = add_directional_targets(df)
# cutoff datapoints without matching wiki csv data
cutoff = pd.Timestamp("2026-04-09", tz="UTC")
df = df.loc[df["timestamp"] < cutoff].copy()

windows = {
    "1h": 1,
    "3h": 3,
    "6h": 6,
    # "12h": 12,
    # "24h": 24,
}

wiki_cols = list(features.columns)[1:]

new_cols = {}

# returns must be within market
new_cols["returns"] = (
    df.groupby("market_slug")["price"]
    .pct_change()
)

for label, hours in windows.items():
    g = df.groupby("market_slug", group_keys=False)

    # price targets / volatility
    new_cols[f"delta_price_{label}"] = (
        g["price"].shift(-hours) - df["price"]
    )

    new_cols[f"abs_delta_price_move_{label}"] = (
        new_cols[f"delta_price_{label}"].abs()
    )

    new_cols[f"volatility_{label}"] = (
        new_cols["returns"]
        .groupby(df["market_slug"])
        .rolling(hours, min_periods=1)
        .std()
        .reset_index(level=0, drop=True)
    )

    # wiki rolling features
    for col in wiki_cols:
        rolled = (
            df[col]
            .groupby(df["market_slug"])
            .rolling(hours, min_periods=1)
            .mean()
            .reset_index(level=0, drop=True)
        )

        new_cols[f"{col}_{label}"] = rolled
        new_cols[f"{col}_{label}_diff"] = (
            rolled.groupby(df["market_slug"]).diff()
        )

df = pd.concat([df, pd.DataFrame(new_cols, index=df.index)], axis=1).copy()
print(df.shape)
df.tail()

wiki_cols

In [ ]:
import matplotlib.pyplot as plt
df = df.loc[:, ~df.columns.duplicated()].copy()
features_to_plot = [
    "new_editors_6h",
    "unique_editors_6h",
    "edits_6h",
    "total_comment_len_6h",
]
agg = (
    df.sort_values("timestamp")
      .groupby("timestamp")[features_to_plot + ["volatility_6h"]]
      .mean()
      .reset_index()
)

fig, ax1 = plt.subplots(figsize=(14, 7))

for feature in features_to_plot:
    ax1.plot(agg["timestamp"], agg[feature], label=feature, linewidth=1.5)

ax1.set_xlabel("Timestamp")
ax1.set_ylabel("Average wiki feature value")
ax1.set_yscale("log")
ax1.legend(loc="upper left")
ax1.grid(alpha=0.3, which="both")

ax2 = ax1.twinx()
ax2.plot(agg["timestamp"], agg["volatility_6h"], label="volatility_6h", color="black", linewidth=2)
ax2.set_ylabel("Average volatility_6h")
ax2.set_yscale("log")
ax2.legend(loc="upper right")

plt.title("Average Wiki Features and 6h Volatility Over Time")
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

df = df.loc[:, ~df.columns.duplicated()].copy()

features_to_plot = [col + "_6h" for col in wiki_cols]

max_lag = 24
target_horizon = 6

plt.figure(figsize=(12, 6))

for feature in features_to_plot:
    market_corrs = {}

    for slug, g in df.groupby("market_slug"):
        g = g.sort_values("timestamp").copy()

        target = g["volatility_6h"].shift(-target_horizon)

        corrs = []
        for lag in range(max_lag):
            x = g[feature].shift(lag)
            corrs.append(x.corr(target))

        market_corrs[slug] = corrs

    corr_df = pd.DataFrame(market_corrs)
    avg_corr = corr_df.mean(axis=1)

    # NEGATIVE time axis
    time_to_event = -(np.arange(max_lag) + target_horizon)

    plt.plot(
        time_to_event,
        avg_corr,
        marker="o",
        linewidth=2,
        label=feature
    )

    # mark strongest signal
    best_idx = np.argmax(avg_corr)
    best_time = -(best_idx + target_horizon)
    plt.axvline(best_time, linestyle="--", alpha=0.2)

plt.axhline(0, linestyle="--", alpha=0.5)

plt.xlabel("Time relative to volatility event (hours)")
plt.ylabel("Average correlation")
plt.title("Lead-Lag Relationship (Event-Centered View)")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# remove duplicate dataframe columns from repeated cell runs
df = df.loc[:, ~df.columns.duplicated()].copy()

### Text Processing with nmf

In [ ]:
# prepare text


rev_text = all_revisions.copy()
rev_text["timestamp"] = pd.to_datetime(rev_text["timestamp"], utc=True)
rev_text["hour"] = rev_text["timestamp"].dt.floor("h")

rev_text["comment"] = rev_text["comment"].fillna("").astype(str)

rev_text["diff_html"] = rev_text.get("diff_html", "").fillna("").astype(str)
rev_text["diff_clean"] = rev_text["diff_html"].apply(extract_added_text)
rev_text["diff_clean"] = rev_text["diff_clean"].apply(remove_wiki_usernames)
rev_text["is_talk"] = rev_text["page_title"].str.startswith("Talk:")

# optional: spike-hour filter
df["edit_spike"] = (
    df.groupby("market_slug")["edits"]
    .transform(lambda s: s > s.quantile(0.9))
)

df = df.drop(
    columns=[c for c in df.columns if c.startswith(("diff_nmf_", "comment_nmf_", "tfidf_nmf_", "article_", "talk_"))],
    errors="ignore"
)

spike_hours = df.loc[df["edit_spike"], "timestamp"].unique()
rev_text_model = rev_text[rev_text["hour"].isin(spike_hours)].copy()

# build separate nmf features for each category
text_cols = []
configs = [
    {"name": "article_diff", "is_talk": False, "text_col": "diff_clean", "prefix": "article_diff"},
    {"name": "article_comment", "is_talk": False, "text_col": "comment", "prefix": "article_comment"},
    {"name": "talk_diff", "is_talk": True, "text_col": "diff_clean", "prefix": "talk_diff"},
    {"name": "talk_comment", "is_talk": True, "text_col": "comment", "prefix": "talk_comment"},
]
text_cols = []
text_models = {}

for config in configs:
    subset = rev_text_model[rev_text_model["is_talk"] == config["is_talk"]].copy()

    nmf_df, vectorizer_, nmf_, terms_, cols = build_text_nmf_features_safe(
        subset,
        text_col=config["text_col"],
        prefix=config["prefix"],
        tokenizer=spacy_tokenizer,
        ngram_range=(1, 3),  # start less sparse than (1,3)
        min_df=2,            # important for small subsets
    )

    if cols:
        rev_text_model = pd.concat([rev_text_model, nmf_df], axis=1)
        text_cols.extend(cols)

        text_models[config["prefix"]] = {
            "vectorizer": vectorizer_,
            "nmf": nmf_,
            "terms": terms_,
            "cols": cols,
        }

print("Final text cols:", text_cols)
tfidf_hourly = (
    rev_text_model.groupby("hour")[text_cols]
    .mean()
    .reset_index()
    .rename(columns={"hour": "timestamp"})
)
df = df.loc[:, ~df.columns.duplicated()].copy()
df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)

df = pd.merge_asof(
    df.sort_values("timestamp"),
    tfidf_hourly.sort_values("timestamp"),
    on="timestamp",
    direction="backward"
)

df[text_cols] = df[text_cols].fillna(0)
df = df.sort_values(["market_slug", "timestamp"]).copy()

print("Text cols:", text_cols)

In [ ]:
from utils.text.nmf import get_nmf_topic_keywords, get_nmf_topic_strengths


max_lag = 24
target_horizon = 6
time_to_event = -(np.arange(max_lag) + target_horizon)



all_strengths = np.concatenate([
    get_nmf_topic_strengths(model_obj["nmf"])
    for model_obj in text_models.values()
])

strength_min = all_strengths.min()
strength_max = all_strengths.max()

def alpha_from_strength(strength, min_alpha=0.25, max_alpha=1.0):
    """Map a topic strength to a matplotlib alpha value.

    Scales strength linearly to the [min_alpha, max_alpha] range based on
    the global strength_min and strength_max values.

    Args:
        strength (float): Topic strength to convert.
        min_alpha (float): Minimum alpha for the weakest topic.
        max_alpha (float): Maximum alpha for the strongest topic.

    Returns:
        float: Alpha value to use for plotting.
    """
    if strength_max == strength_min:
        return max_alpha
    scaled = (strength - strength_min) / (strength_max - strength_min)
    return min_alpha + scaled * (max_alpha - min_alpha)

plt.figure(figsize=(14, 7))

for model_name, model_obj in text_models.items():
    cols = model_obj["cols"]
    nmf = model_obj["nmf"]
    terms = model_obj["terms"]
    strengths = get_nmf_topic_strengths(nmf)

    for i, col in enumerate(cols):
        market_corrs = {}

        for slug, g in df.groupby("market_slug"):
            g = g.sort_values("timestamp").copy()
            target = g["volatility_6h"].shift(-target_horizon)

            corrs = []
            for lag in range(max_lag):
                x = g[col].shift(lag)
                corrs.append(x.corr(target))

            market_corrs[slug] = corrs

        corr_df_temp = pd.DataFrame(market_corrs)
        avg_corr = corr_df_temp.mean(axis=1)

        label_words = ", ".join(
            get_nmf_topic_keywords(nmf, terms, i, top_n=3)["term"]
        )
        alpha = alpha_from_strength(strengths[i])

        plt.plot(
            time_to_event,
            avg_corr,
            marker="o",
            linewidth=1.5,
            alpha=alpha,
            label=f"{model_name}: {label_words} | strength={strengths[i]:.3f}"
        )

plt.axhline(0, linestyle="--", alpha=0.5)

plt.title("Average Lead–Lag Correlation for Article/Talk Text Topics vs Volatility")
plt.xlabel("Time before volatility event (hours)")
plt.ylabel("Average correlation")

plt.legend(
    title="Top latent text topics; darker = higher topic strength",
    fontsize=8,
    title_fontsize=9,
    loc="upper left",
    bbox_to_anchor=(1.02, 1)
)

plt.tight_layout()
plt.show()

In [ ]:

# choose channel + topic
channel = "talk_comment"   # options: article_diff, article_comment, talk_diff, talk_comment
topic_idx = 0

model_obj = text_models[channel]

nmf = model_obj["nmf"]
terms = model_obj["terms"]

pos = get_nmf_topic_keywords(
    nmf,
    terms,
    topic_idx,
    top_n=15
)

print(f"Channel: {channel}")
print(f"Topic: {topic_idx}")

print("\nPositive side:")
print(pos.to_string(index=False))

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

df = df.copy()
df = df.sort_values(["market_slug", "timestamp"]).copy()

# =====================================================
# KERNELS
# =====================================================

# short-term reaction kernel (~4–8h)
short_kernel = np.array([
    0, 0, 0,
    0.15, 0.22, 0.26, 0.22, 0.15
])

short_kernel = short_kernel / short_kernel.sum()

# long-term buildup kernel (~4–24h)
long_kernel = np.array([
    0, 0, 0, 0,
    0.03, 0.04, 0.05,
    0.06, 0.07, 0.08,
    0.08, 0.08, 0.08,
    0.07, 0.06, 0.05,
    0.05, 0.04, 0.04,
    0.03, 0.03, 0.02,
    0.02, 0.02
])

long_kernel = long_kernel / long_kernel.sum()

# =====================================================
# BUILD SIGNAL FEATURES
# =====================================================

base_cols = text_cols + [
    "edits_6h",
    "new_editors_6h",
    "total_comment_len_6h"
]

signal_cols = []

for col in base_cols:

    # -----------------------------
    # SHORT SIGNAL
    # -----------------------------
    short_parts = []

    for lag, w in enumerate(short_kernel, start=1):
        short_parts.append(
            w * df.groupby("market_slug")[col].shift(lag)
        )

    short_col = f"{col}_short_signal"

    df[short_col] = sum(short_parts)

    signal_cols.append(short_col)

    # -----------------------------
    # LONG SIGNAL
    # -----------------------------
    long_parts = []

    for lag, w in enumerate(long_kernel, start=1):
        long_parts.append(
            w * df.groupby("market_slug")[col].shift(lag)
        )

    long_col = f"{col}_long_signal"

    df[long_col] = sum(long_parts)

    signal_cols.append(long_col)

# =====================================================
# FEATURE ENGINEERING
# =====================================================

feature_cols = signal_cols.copy()

# interaction features
for col in text_cols:

    short_interaction = f"{col}_short_x_new_editors"
    long_interaction = f"{col}_long_x_new_editors"

    df[short_interaction] = (
        df[f"{col}_short_signal"]
        * df["new_editors_6h_short_signal"]
    )

    df[long_interaction] = (
        df[f"{col}_long_signal"]
        * df["new_editors_6h_long_signal"]
    )

    feature_cols.append(short_interaction)
    feature_cols.append(long_interaction)

# aggregate intensity features
short_text_signals = [
    f"{col}_short_signal"
    for col in text_cols
]

long_text_signals = [
    f"{col}_long_signal"
    for col in text_cols
]

df["short_text_intensity"] = (
    df[short_text_signals]
    .abs()
    .sum(axis=1)
)

df["long_text_intensity"] = (
    df[long_text_signals]
    .abs()
    .sum(axis=1)
)

feature_cols.append("short_text_intensity")
feature_cols.append("long_text_intensity")

# =====================================================
# TARGET
# =====================================================

df["target"] = (
    df.groupby("market_slug")["volatility_6h"]
    .shift(-1)
)

# =====================================================
# CLEAN MODEL FRAME
# =====================================================

model_df = pd.concat(
    [
        df[["market_slug", "timestamp"]],
        df[feature_cols],
        df["target"]
    ],
    axis=1
).dropna()

model_df = model_df.sort_values("timestamp")

# =====================================================
# TRAIN / TEST SPLIT
# =====================================================

split = int(len(model_df) * 0.7)

train_df = model_df.iloc[:split].copy()
test_df = model_df.iloc[split:].copy()

threshold = train_df["target"].quantile(0.9)

train_df["big_move"] = (
    train_df["target"] > threshold
).astype(int)

test_df["big_move"] = (
    test_df["target"] > threshold
).astype(int)

print("Train class counts:")
print(train_df["big_move"].value_counts())

print("\nTest class counts:")
print(test_df["big_move"].value_counts())

# =====================================================
# FEATURES / TARGETS
# =====================================================

X_train = train_df[feature_cols]
X_test = test_df[feature_cols]

y_train = train_df["big_move"]
y_test = test_df["big_move"]

# =====================================================
# SCALE
# =====================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# =====================================================
# MODEL
# =====================================================

model = LogisticRegression(
    max_iter=1000,
    C=0.5,
    penalty="l1",
    solver="liblinear"
)

model.fit(X_train_scaled, y_train)

# =====================================================
# EVALUATION
# =====================================================

y_pred_prob = model.predict_proba(X_test_scaled)[:, 1]

auc = roc_auc_score(y_test, y_pred_prob)

print("\nAUC:", auc)

# =====================================================
# FEATURE IMPORTANCE
# =====================================================

coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coef": model.coef_[0]
}).sort_values("coef", ascending=False)

print("\nTop positive:")
print(coef_df.head(15))

print("\nTop negative:")
print(coef_df.tail(15))

In [ ]:
y_train_shuffled = y_train.sample(frac=1, random_state=42)

model.fit(X_train_scaled, y_train_shuffled)
y_pred_prob_shuffled = model.predict_proba(X_test_scaled)[:, 1]

print("Shuffled AUC:", roc_auc_score(y_test, y_pred_prob_shuffled))

In [ ]:
shuffle_aucs = []

for seed in tqdm(range(50)):
    y_perm = y_train.sample(frac=1, random_state=seed)
    m = LogisticRegression(max_iter=1000)
    m.fit(X_train_scaled, y_perm)
    pred = m.predict_proba(X_test_scaled)[:, 1]
    shuffle_aucs.append(roc_auc_score(y_test, pred))

print("Real AUC:", auc)
print("Shuffle mean:", np.mean(shuffle_aucs))
print("Shuffle max:", np.max(shuffle_aucs))

#### nmf vizs

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(13, 4.8))
ax.axis("off")

def draw_matrix(ax, x, y, w, h, label, shape, color="white"):
    rect = plt.Rectangle(
        (x, y),
        w,
        h,
        facecolor=color,
        edgecolor="black",
        linewidth=1.5
    )
    ax.add_patch(rect)

    ax.text(
        x + w/2,
        y + h/2,
        label,
        ha="center",
        va="center",
        fontsize=18,
        weight="bold"
    )

    ax.text(
        x + w/2,
        y - 0.15,
        shape,
        ha="center",
        va="top",
        fontsize=11
    )

# -----------------------------
# Draw matrices
# -----------------------------
draw_matrix(
    ax,
    x=0.2,
    y=1.0,
    w=1.6,
    h=2.0,
    label="X",
    shape="documents × terms\nTF-IDF matrix",
    color="#f2f2f2"
)

ax.text(2.1, 2.0, "≈", fontsize=24, ha="center", va="center")

draw_matrix(
    ax,
    x=2.5,
    y=1.0,
    w=1.2,
    h=2.0,
    label="U",
    shape="documents × topics\nunscaled topic directions",
    color="#e8f4ff"
)

draw_matrix(
    ax,
    x=4.0,
    y=1.5,
    w=1.0,
    h=1.0,
    label="Σ",
    shape="topics × topics\nsingular values",
    color="#fff4cc"
)

draw_matrix(
    ax,
    x=5.3,
    y=1.0,
    w=2.0,
    h=2.0,
    label="Vᵀ",
    shape="topics × terms\ntopic-word weights",
    color="#eaffea"
)

# multiplication symbols
ax.text(3.85, 2.0, "×", fontsize=20, ha="center", va="center")
ax.text(5.15, 2.0, "×", fontsize=20, ha="center", va="center")

# -----------------------------
# Highlight what sklearn returns
# -----------------------------
highlight = plt.Rectangle(
    (2.38, 0.82),
    2.75,
    2.35,
    facecolor="none",
    edgecolor="#1f77b4",
    linewidth=2.2,
    linestyle="--"
)
ax.add_patch(highlight)

# -----------------------------
# Annotations
# -----------------------------
ax.text(
    0.2,
    3.35,
    "Text Frequency Matrix",
    fontsize=12,
    ha="left"
)

ax.text(
    2.5,
    3.35,
    "Documents ---> combinations of topics",
    fontsize=12,
    ha="left",
    color="#1f77b4",
    weight="bold"
)

ax.text(
    5.3,
    3.35,
    "Topics ---> combinations of words",
    fontsize=12,
    ha="left",
    color="#2ca02c",
    weight="bold"
)

ax.set_xlim(0, 7.8)
ax.set_ylim(0.1, 3.9)

plt.tight_layout()
plt.show()

In [ ]:


# -------------------------------------------------
# SELECT TOP FEATURES
# -------------------------------------------------
top_n = 15

top_pos = coef_df.sort_values(
    "coef",
    ascending=False
).head(top_n)

top_neg = coef_df.sort_values(
    "coef",
    ascending=True
).head(top_n)

plot_df = pd.concat([
    top_neg,
    top_pos
]).drop_duplicates().copy()

plot_df = plot_df.sort_values("coef")

plot_df["label"] = plot_df["feature"].apply(
    lambda x: replace_nmf_with_keywords(
        x,
        text_models,
        top_n=3
    )
)

# -------------------------------------------------
# COLOR MAPPING
# -------------------------------------------------
coef_max = np.max(np.abs(plot_df["coef"]))

norm = mcolors.TwoSlopeNorm(
    vmin=-coef_max,
    vcenter=0,
    vmax=coef_max
)

cmap = cm.RdBu

colors = [
    cmap(norm(v))
    for v in plot_df["coef"]
]

# -------------------------------------------------
# PLOT
# -------------------------------------------------
fig, ax = plt.subplots(figsize=(18, 10))

y_positions = np.arange(len(plot_df))

ax.barh(
    y=y_positions,
    width=plot_df["coef"],
    color=colors,
    alpha=0.9
)

# -------------------------------------------------
# REMOVE Y TICKS
# -------------------------------------------------
ax.set_yticks([])

# -------------------------------------------------
# CENTER LINE
# -------------------------------------------------
ax.axvline(
    0,
    color="black",
    linestyle="--",
    linewidth=1.2
)

# -------------------------------------------------
# LABEL ANNOTATIONS
# -------------------------------------------------
label_offset = coef_max * 0.03

for y, coef, label in zip(
    y_positions,
    plot_df["coef"],
    plot_df["label"]
):

    # positive coefficients (blue bars)
    if coef > 0:
        ax.text(
            -label_offset,
            y,
            label,
            ha="right",
            va="center",
            fontsize=9
        )

    # negative coefficients (red bars)
    else:
        ax.text(
            label_offset,
            y,
            label,
            ha="left",
            va="center",
            fontsize=9
        )

# -------------------------------------------------
# TITLES
# -------------------------------------------------
ax.set_title(
    "Most Predictive Wikipedia-Derived Features\n"
    "Latent Topics Replaced with Top Keywords",
    fontsize=14
)

ax.set_xlabel(
    "Logistic regression coefficient",
    fontsize=12
)

# -------------------------------------------------
# CLEAN LOOK
# -------------------------------------------------
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["left"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# -----------------------------
# CONFIG
# -----------------------------
slug = "will-the-us-invade-iran-before-2027"
vol_col = "volatility_6h"

smooth_window = 6
vol_quantile = 0.9

# -----------------------------
# SELECT ONE MARKET
# -----------------------------
g = (
    df[df["market_slug"] == slug]
    .sort_values("timestamp")
    .copy()
)

# -----------------------------
# USE EXISTING TRAINED MODEL
# -----------------------------
plot_df = g[["timestamp", vol_col] + feature_cols].dropna().copy()

X_plot = plot_df[feature_cols]
X_plot_scaled = scaler.transform(X_plot)

plot_df["pred_prob"] = model.predict_proba(X_plot_scaled)[:, 1]
plot_df["model_score"] = model.decision_function(X_plot_scaled)

# -----------------------------
# DISPLAY VOLATILITY ONLY
# -----------------------------
plot_df["vol_smooth"] = (
    plot_df[vol_col]
    .rolling(smooth_window)
    .mean()
)

# Actual top-decile volatility for visualization only
vol_threshold = plot_df["vol_smooth"].quantile(vol_quantile)
plot_df["actual_top_decile_vol"] = plot_df["vol_smooth"] > vol_threshold

plot_df = plot_df.dropna(subset=["vol_smooth"])

# -----------------------------
# NEWS EVENTS
# -----------------------------
news_events = [
    {"time": "2026-04-02 12:00:00+00:00", "label": "Trump: hit Iran 'very hard'"},
    {"time": "2026-04-05 00:00:00+00:00", "label": "US jets reportedly shot down"},
    {"time": "2026-04-07 00:00:00+00:00", "label": "Ceasefire discussions"},
]

# -----------------------------
# PLOT
# -----------------------------
fig, ax1 = plt.subplots(figsize=(14, 6))

ax1.plot(
    plot_df["timestamp"],
    plot_df["vol_smooth"],
    color="black",
    linewidth=2,
    label="Smoothed realized 6h volatility"
)

actual_spikes = plot_df["actual_top_decile_vol"]

ax1.scatter(
    plot_df.loc[actual_spikes, "timestamp"],
    plot_df.loc[actual_spikes, "vol_smooth"],
    color="red",
    s=40,
    label="Actual top-decile volatility"
)

ax1.set_ylabel("Smoothed realized volatility")
ax1.set_xlabel("Time (UTC)")
ax1.grid(alpha=0.3)

ax2 = ax1.twinx()

ax2.plot(
    plot_df["timestamp"],
    plot_df["pred_prob"],
    color="orange",
    linewidth=2,
    label="Panel model predicted probability"
)

ax2.fill_between(
    plot_df["timestamp"],
    0,
    plot_df["pred_prob"],
    color="orange",
    alpha=0.25
)

# dynamic probability scale
pmax = plot_df["pred_prob"].max()
ax2.set_ylim(0, max(0.05, pmax * 1.0))
ax2.set_ylabel("Predicted probability")

# News event markers
for i, event in enumerate(news_events):
    t = pd.to_datetime(event["time"], utc=True)

    ax1.axvline(
        x=t,
        color="blue",
        linestyle="--",
        linewidth=1.3,
        alpha=0.65
    )

    y_frac = 0.96 if i % 2 == 0 else 0.86

    ax1.annotate(
        event["label"],
        xy=(t, y_frac),
        xycoords=("data", "axes fraction"),
        xytext=(8, 0),
        textcoords="offset points",
        rotation=90,
        va="top",
        ha="left",
        fontsize=8,
        color="blue",
        bbox=dict(
            boxstyle="round,pad=0.25",
            fc="white",
            ec="blue",
            alpha=0.75,
            linewidth=0.7
        ),
        clip_on=True,
        zorder=10
    )

plt.title(
    f"Panel Model Predictions vs Realized Volatility\n"
    f"Market: {slug}"
)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()

ax1.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="upper left"
)

plt.xlim(plot_df["timestamp"].min(), plot_df["timestamp"].max())
plt.tight_layout()
plt.show()

### Regression

#### Linear Regression

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

df = df.copy()
df = df.sort_values(["market_slug", "timestamp"]).copy()

# =====================================================
# KERNELS
# =====================================================

# short-term reaction kernel (~4–8h)
short_kernel = np.array([
    0, 0, 0,
    0.15, 0.22, 0.26, 0.22, 0.15
])

short_kernel = short_kernel / short_kernel.sum()

# long-term buildup kernel (~4–24h)
long_kernel = np.array([
    0, 0, 0, 0,
    0.03, 0.04, 0.05,
    0.06, 0.07, 0.08,
    0.08, 0.08, 0.08,
    0.07, 0.06, 0.05,
    0.05, 0.04, 0.04,
    0.03, 0.03, 0.02,
    0.02, 0.02
])

long_kernel = long_kernel / long_kernel.sum()

# =====================================================
# BUILD SIGNAL FEATURES
# =====================================================

base_cols = text_cols + [
    "edits_6h",
    "new_editors_6h",
    "total_comment_len_6h"
]

signal_cols = []

for col in base_cols:

    # -----------------------------
    # SHORT SIGNAL
    # -----------------------------
    short_parts = []

    for lag, w in enumerate(short_kernel, start=1):
        short_parts.append(
            w * df.groupby("market_slug")[col].shift(lag)
        )

    short_col = f"{col}_short_signal"

    df[short_col] = sum(short_parts)

    signal_cols.append(short_col)

    # -----------------------------
    # LONG SIGNAL
    # -----------------------------
    long_parts = []

    for lag, w in enumerate(long_kernel, start=1):
        long_parts.append(
            w * df.groupby("market_slug")[col].shift(lag)
        )

    long_col = f"{col}_long_signal"

    df[long_col] = sum(long_parts)

    signal_cols.append(long_col)

# =====================================================
# FEATURE ENGINEERING
# =====================================================

feature_cols = signal_cols.copy()

# interaction features
for col in text_cols:

    short_interaction = f"{col}_short_x_new_editors"
    long_interaction = f"{col}_long_x_new_editors"

    df[short_interaction] = (
        df[f"{col}_short_signal"]
        * df["new_editors_6h_short_signal"]
    )

    df[long_interaction] = (
        df[f"{col}_long_signal"]
        * df["new_editors_6h_long_signal"]
    )

    feature_cols.append(short_interaction)
    feature_cols.append(long_interaction)

# aggregate intensity features
short_text_signals = [
    f"{col}_short_signal"
    for col in text_cols
]

long_text_signals = [
    f"{col}_long_signal"
    for col in text_cols
]

df["short_text_intensity"] = (
    df[short_text_signals]
    .abs()
    .sum(axis=1)
)

df["long_text_intensity"] = (
    df[long_text_signals]
    .abs()
    .sum(axis=1)
)

feature_cols.append("short_text_intensity")
feature_cols.append("long_text_intensity")

# =====================================================
# TARGET
# =====================================================

df["target"] = (
    df.groupby("market_slug")["volatility_6h"]
    .shift(-1)
)

# =====================================================
# CLEAN MODEL FRAME
# =====================================================

model_df = pd.concat(
    [
        df[["market_slug", "timestamp"]],
        df[feature_cols],
        df["target"]
    ],
    axis=1
).dropna()

model_df = model_df.sort_values("timestamp")

# =====================================================
# TRAIN / TEST SPLIT
# =====================================================

split = int(len(model_df) * 0.7)

train_df = model_df.iloc[:split].copy()
test_df = model_df.iloc[split:].copy()

threshold = train_df["target"].quantile(0.9)

# =====================================================
# FEATURES / TARGETS
# =====================================================

X_train = train_df[feature_cols]
X_test = test_df[feature_cols]

y_train = train_df["target"]
y_test = test_df["target"]

# =====================================================
# SCALE
# =====================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# =====================================================
# MODEL
# =====================================================

model = LinearRegression()

model.fit(X_train_scaled, y_train)

# =====================================================
# EVALUATION
# =====================================================

r2 = model.score(X_test_scaled, y_test)
print("\nR2:", r2)

# =====================================================
# FEATURE IMPORTANCE
# =====================================================

coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coef": model.coef_[0]
}).sort_values("coef", ascending=False)

print("\nTop positive:")
print(coef_df.head(15))

print("\nTop negative:")
print(coef_df.tail(15))

#### ARIMA

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error, mean_absolute_error

# -----------------------------
# CONFIG
# -----------------------------
slug = "will-the-us-invade-iran-before-2027"
vol_col = "volatility_6h"

# -----------------------------
# SELECT ONE MARKET
# -----------------------------
g = (
    df[df["market_slug"] == slug]
    .sort_values("timestamp")
    .copy()
)

# -----------------------------
# BUILD TIME SERIES
# -----------------------------
y = (
    g[["timestamp", vol_col]]
    .dropna()
    .set_index("timestamp")[vol_col]
)

# optional: stabilize volatility
y = np.log(y + 1e-6)

# -----------------------------
# TRAIN / TEST SPLIT
# -----------------------------
split = int(len(y) * 0.7)

y_train = y.iloc[:split]
y_test = y.iloc[split:]

# -----------------------------
# FIT ARIMA BASELINE
# -----------------------------
# Start with ARIMA(1,0,1): ARMA-style volatility persistence model
arima = ARIMA(
    y_train,
    order=(1, 0, 1)
)

res = arima.fit()

print(res.summary())

# -----------------------------
# FORECAST TEST PERIOD
# -----------------------------
forecast = res.forecast(steps=len(y_test))

# align index
forecast.index = y_test.index

# -----------------------------
# EVALUATE
# -----------------------------
rmse = np.sqrt(mean_squared_error(y_test, forecast))
mae = mean_absolute_error(y_test, forecast)

print("\nARIMA baseline:")
print("RMSE:", rmse)
print("MAE:", mae)

# -----------------------------
# PLOT
# -----------------------------
plt.figure(figsize=(14, 6))

plt.plot(
    y_train.index,
    y_train,
    label="Train log volatility",
    alpha=0.7
)

plt.plot(
    y_test.index,
    y_test,
    label="Actual test log volatility",
    color="black",
    linewidth=2
)

plt.plot(
    forecast.index,
    forecast,
    label="ARIMA forecast",
    color="orange",
    linewidth=2
)

plt.axvline(
    y_test.index.min(),
    linestyle="--",
    color="gray",
    label="Train/test split"
)

plt.title(f"ARIMA Baseline Forecast for Log Volatility\nMarket: {slug}")
plt.xlabel("Time")
plt.ylabel("Log 6h volatility")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

#### SARIMAX

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_squared_error, mean_absolute_error

# -------------------------------------------------
# CONFIG
# -------------------------------------------------
slug = "will-the-us-invade-iran-before-2027"

vol_col = "volatility_6h"

selected_features = [
    "edits_6h",
    "total_comment_len_6h"
]

# -------------------------------------------------
# SELECT MARKET
# -------------------------------------------------
g = df[df["market_slug"] == slug].sort_values("timestamp").copy()

# -------------------------------------------------
# BUILD DATASET
# -------------------------------------------------
model_df = g[["timestamp", vol_col] + selected_features].dropna()

model_df = model_df.set_index("timestamp")

# optional stabilization
model_df["log_vol"] = np.log(model_df[vol_col] + 1e-6)

# -------------------------------------------------
# TARGET / EXOGENOUS
# -------------------------------------------------
y = model_df["log_vol"]

X = model_df[selected_features]

# -------------------------------------------------
# TRAIN / TEST SPLIT
# -------------------------------------------------
split = int(len(model_df) * 0.7)

y_train = y.iloc[:split]
y_test = y.iloc[split:]

X_train = X.iloc[:split]
X_test = X.iloc[split:]

# -------------------------------------------------
# FIT SARIMAX
# -------------------------------------------------
model = SARIMAX(
    endog=y_train,
    exog=X_train,
    # ARIMA(1,0,1)
    order=(1, 0, 1),
    enforce_stationarity=False,
    enforce_invertibility=False,
)

results = model.fit(disp=False)

print(results.summary())

# -------------------------------------------------
# FORECAST
# -------------------------------------------------
forecast = results.forecast(steps=len(y_test), exog=X_test)

forecast.index = y_test.index

# -------------------------------------------------
# EVALUATE
# -------------------------------------------------
rmse = np.sqrt(mean_squared_error(y_test, forecast))

mae = mean_absolute_error(y_test, forecast)

print("\nSARIMAX Results")
print("RMSE:", rmse)
print("MAE:", mae)

# -------------------------------------------------
# PLOT
# -------------------------------------------------
plt.figure(figsize=(14, 6))

plt.plot(y_train.index, y_train, alpha=0.5, label="Train log volatility")

plt.plot(
    y_test.index, y_test, color="black", linewidth=2, label="Actual test log volatility"
)

plt.plot(
    forecast.index, forecast, color="orange", linewidth=2, label="SARIMAX forecast"
)

plt.axvline(y_test.index.min(), linestyle="--", color="gray", alpha=0.7)

plt.title(f"SARIMAX Forecast with Wikipedia Signals\n{slug}")

plt.xlabel("Time")
plt.ylabel("Log volatility")

plt.legend()
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()